In [79]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [81]:
arbres = pd.read_csv("train.csv", index_col = 'ID_ARBRE')
prev = pd.read_csv("prev.csv", index_col = 'ID_ARBRE')
benchmark = pd.read_csv("benchmark.csv", index_col = 'ID_ARBRE')

In [83]:
x = arbres[["type_sol", "classe_age", "classe_hauteur", "classe_circonference", "port_arbre", "vigueur_pousse", "plaie_collet", "fissure_tronc", "plaie_tronc", "fissure_houppier", "bois_mort_houppier", "plaie_houppier", "esperance_maintien"]]
y = arbres[["classification_diagnostic"]]

In [85]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size = 0.2)

In [87]:
def loi_jointe(data, X1, Y):
    ux = np.unique(data[X1])
    uy = np.unique(data[Y])
    x = data[X1]
    y = data[Y]
    tab_loi = np.zeros((len(uy), len(ux)))
    for c in range(len(uy)):
        for j in range(len(ux)):
            tab_loi[c, j] = np.mean((x == ux[j]) * (y == uy[c])) + 0.0001
    return tab_loi

In [89]:
joint = loi_jointe(arbres, "classe_age", "classification_diagnostic")
joint

array([[1.06717647e-01, 1.00000000e-04, 1.41644118e-01, 9.20117647e-02],
       [3.65908824e-01, 3.77647059e-03, 7.73058824e-02, 1.32452941e-01],
       [2.39970588e-02, 1.00000000e-04, 3.77647059e-03, 2.95117647e-02],
       [9.29117647e-03, 1.00000000e-04, 1.93823529e-03, 1.93823529e-03],
       [9.29117647e-03, 1.00000000e-04, 1.00000000e-04, 1.93823529e-03]])

In [91]:
def classifieur(data, X, X_val, Y):
    tab_joint = loi_jointe(data, X, Y) # Probas Jointes
    uy = np.unique(data[Y]) # np.unique de Y
    ux = np.unique(data[X]) #np.unique de X
    num_val = np.argmax(ux == X_val) # On renvoie l'index de X_val dans le np.unique de X
    
    pysx = []
    for c in range(len(uy)):
        pysx.append(tab_joint[c, num_val])
    return uy[np.argmax(pysx)]

In [93]:
np.unique(arbres["classe_age"])

array(['A', 'AM', 'J', 'JA'], dtype=object)

In [95]:
classifieur(arbres, "classe_age", "JA", "classification_diagnostic")

'C2'

In [97]:
df = prev["classe_age"]
len(df)

137

In [99]:
previsions = [[classifieur(arbres, "classe_age", df.iloc[i], "classification_diagnostic") for i in range(len(df))] for c in x.columns]

In [141]:
vote = []

for index in range(len(df)):
    uy = np.unique(y)
    ligne_i = []
    for prev_col in range(len(previsions)):
        ligne_i.append(previsions[prev_col][index])
    unique_elements, counts = np.unique(ligne_i, return_counts=True)
    vote.append(unique_elements[np.argmax(counts)])

In [143]:
vote

['C2',
 'C2',
 'C2',
 'C1',
 'C2',
 'C2',
 'C2',
 'C1',
 'C2',
 'C2',
 'C2',
 'C1',
 'C1',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C1',
 'C2',
 'C2',
 'C2',
 'C1',
 'C2',
 'C1',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C1',
 'C2',
 'C2',
 'C2',
 'C2',
 'C1',
 'C1',
 'C2',
 'C2',
 'C2',
 'C1',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C1',
 'C2',
 'C2',
 'C2',
 'C1',
 'C2',
 'C2',
 'C1',
 'C2',
 'C2',
 'C1',
 'C2',
 'C2',
 'C2',
 'C1',
 'C1',
 'C1',
 'C2',
 'C1',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C1',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C1',
 'C1',
 'C2',
 'C1',
 'C1',
 'C2',
 'C2',
 'C2',
 'C2',
 'C1',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C1',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C2',
 'C1',
 'C2',
 'C2',
 'C2',
 'C1',
 'C2',
 'C2',
 'C2',
 'C2',
 'C1']

In [123]:
unique_elements, counts = np.unique(previsions[0], return_counts=True)
most_frequent = unique_elements[np.argmax(counts)]

print("L'élément le plus fréquent est :", most_frequent)

L'élément le plus fréquent est : C2


In [156]:
sub = pd.DataFrame({'classification_diagnostic': vote})
sub.index.name = "ID_ARBRE"
id_arbre = df.index  # On récupère l'index chez x2
sub["ID_ARBRE"] = id_arbre  # Ajoute l'index comme colonne
sub = sub.set_index("ID_ARBRE")  # On définie 'ID_ARBRE' en tant qu'index

In [158]:
sub.head()

,classification_diagnostic
ID_ARBRE,
559,C2
560,C2
561,C2
562,C1
563,C2


In [160]:
sub.to_csv("submition20.csv")